# QBC Small PINN Landscape Comparison

This notebook compares the exported loss-landscape results for the small `qbc_deep_ensemble` PINN experiment across:

- `SM4`
- `SM6`
- `SM_AVR_GOV`

It is designed to read the compact export bundle written to the repo-tracked `results/pinn_landscape/<experiment_name>/` folder by `tools/pinn/run_qbc_small_loss_landscape.py`.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

RESULTS_ROOT = Path('results') / 'pinn_landscape'
EXPERIMENT_NAME = None  # Set explicitly to override newest-export auto-discovery.
MODEL_ORDER = ['sm4', 'sm6', 'sm_avr_gov']
MODEL_LABELS = {
    'sm4': 'SM4',
    'sm6': 'SM6',
    'sm_avr_gov': 'SM_AVR_GOV',
}

assert RESULTS_ROOT.exists(), f'Results root not found: {RESULTS_ROOT}'

if EXPERIMENT_NAME is None:
    candidates = [path for path in RESULTS_ROOT.iterdir() if path.is_dir()]
    assert candidates, f'No exported experiments found under: {RESULTS_ROOT}'
    EXPORT_ROOT = max(candidates, key=lambda path: path.stat().st_mtime)
    EXPERIMENT_NAME = EXPORT_ROOT.name
else:
    EXPORT_ROOT = RESULTS_ROOT / EXPERIMENT_NAME

assert EXPORT_ROOT.exists(), f'Export root not found: {EXPORT_ROOT}'
print(f'Using exported experiment: {EXPERIMENT_NAME}')
EXPORT_ROOT


In [ ]:
def read_json(path: Path) -> dict:
    return json.loads(path.read_text())

def load_landscape_dir(path: Path) -> dict:
    coords = np.load(path / 'coordinates.npz')
    losses = {}
    for loss_path in sorted(path.glob('loss_*.npy')):
        losses[loss_path.stem.replace('loss_', '')] = np.load(loss_path)
    return {
        'path': path,
        'manifest': read_json(path / 'manifest.json') if (path / 'manifest.json').is_file() else {},
        'coordinates': {key: coords[key] for key in coords.files},
        'losses': losses,
    }

def load_model_bundle(model_key: str) -> dict:
    model_root = EXPORT_ROOT / model_key
    metrics_path = model_root / 'metrics.csv'
    loss_root = model_root / 'loss_landscape'
    landscapes = {}
    if loss_root.is_dir():
        for path in sorted(loss_root.iterdir()):
            if path.is_dir():
                landscapes[path.name] = load_landscape_dir(path)
    return {
        'model_key': model_key,
        'label': MODEL_LABELS.get(model_key, model_key.upper()),
        'root': model_root,
        'metrics': pd.read_csv(metrics_path) if metrics_path.is_file() else None,
        'landscapes': landscapes,
    }

bundles = {model_key: load_model_bundle(model_key) for model_key in MODEL_ORDER if (EXPORT_ROOT / model_key).exists()}
bundles.keys()

In [ ]:
for model_key, bundle in bundles.items():
    print(bundle['label'])
    print('  metrics:', None if bundle['metrics'] is None else bundle['metrics'].shape)
    print('  landscapes:', sorted(bundle['landscapes'].keys()))

In [ ]:
def plot_training_metrics(bundles: dict) -> None:
    available = {k: v for k, v in bundles.items() if v['metrics'] is not None}
    if not available:
        print('No metrics.csv files found in the export bundle.')
        return

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for model_key, bundle in available.items():
        df = bundle['metrics']
        x = df['global_epoch'] if 'global_epoch' in df.columns else np.arange(len(df))
        axes[0].plot(x, df['train_total_loss'], label=bundle['label'])
        if 'val_data_loss' in df.columns:
            axes[1].plot(x, df['val_data_loss'], label=bundle['label'])
        grad_cols = [c for c in df.columns if c.startswith('train_weighted_') and c.endswith('_grad_norm')]
        if grad_cols:
            axes[2].plot(x, df[grad_cols].sum(axis=1), label=bundle['label'])

    axes[0].set_title('Train Total Loss')
    axes[1].set_title('Validation Data Loss')
    axes[2].set_title('Sum of Weighted Gradient Norms')
    for ax in axes:
        ax.set_xlabel('Global epoch')
        ax.set_yscale('log')
        ax.legend()
    plt.tight_layout()

plot_training_metrics(bundles)

In [ ]:
def plot_1d_comparison(bundles: dict, checkpoint_tag: str = 'best_1d', loss_name: str = 'total') -> None:
    fig, ax = plt.subplots(figsize=(8, 5))
    found = False
    for model_key, bundle in bundles.items():
        landscape = bundle['landscapes'].get(checkpoint_tag)
        if landscape is None or loss_name not in landscape['losses']:
            continue
        alpha = landscape['coordinates']['alpha']
        values = landscape['losses'][loss_name]
        ax.plot(alpha, values, marker='o', label=bundle['label'])
        found = True
    if not found:
        print(f'No 1D landscapes found for checkpoint_tag={checkpoint_tag!r}.')
        return
    ax.set_title(f'1D landscape comparison: {checkpoint_tag} | loss={loss_name}')
    ax.set_xlabel('alpha')
    ax.set_ylabel('loss')
    ax.set_yscale('log')
    ax.legend()
    plt.tight_layout()

plot_1d_comparison(bundles, checkpoint_tag='best_1d', loss_name='total')

In [ ]:
def plot_2d_comparison(bundles: dict, checkpoint_tag: str = 'best_2d', loss_name: str = 'total') -> None:
    available = [(k, v) for k, v in bundles.items() if checkpoint_tag in v['landscapes']]
    if not available:
        print(f'No 2D landscapes found for checkpoint_tag={checkpoint_tag!r}.')
        return

    fig, axes = plt.subplots(1, len(available), figsize=(6 * len(available), 5), squeeze=False)
    axes = axes[0]
    for ax, (model_key, bundle) in zip(axes, available):
        landscape = bundle['landscapes'][checkpoint_tag]
        alpha = landscape['coordinates']['alpha']
        beta = landscape['coordinates']['beta']
        values = landscape['losses'][loss_name]
        contour = ax.contourf(alpha, beta, np.log10(values + 1e-16).T, levels=25, cmap='viridis')
        ax.set_title(bundle['label'])
        ax.set_xlabel('alpha')
        ax.set_ylabel('beta')
        fig.colorbar(contour, ax=ax, label=f'log10({loss_name})')
    plt.tight_layout()

plot_2d_comparison(bundles, checkpoint_tag='best_2d', loss_name='total')

In [ ]:
def summarize_minima(bundles: dict, checkpoint_tags: list[str] | None = None) -> pd.DataFrame:
    if checkpoint_tags is None:
        checkpoint_tags = ['best_1d', 'best_2d']
    rows = []
    for model_key, bundle in bundles.items():
        for checkpoint_tag in checkpoint_tags:
            landscape = bundle['landscapes'].get(checkpoint_tag)
            if landscape is None:
                continue
            for loss_name, values in landscape['losses'].items():
                rows.append({
                    'model': bundle['label'],
                    'checkpoint': checkpoint_tag,
                    'loss_name': loss_name,
                    'min_loss': float(np.nanmin(values)),
                    'max_loss': float(np.nanmax(values)),
                })
    return pd.DataFrame(rows)

summary_df = summarize_minima(bundles)
summary_df